In [1]:
import sys
import os
from pathlib import Path
import time

import cv2 as cv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.patches as patches
from joblib import Parallel, delayed

In [2]:
class Sample:
    """Class that contains relevant data of a sample investigated with profilometer"""
    # Conversion factor
    UM2MM = 1e-3
    
    def __init__(self, data_path, time, sample_id):
        self.data_path = data_path
        self.sample_id = sample_id
        
        # Read metadata from CSV
        height_path = data_path / time / "height" / f"{sample_id}.csv"
        with open(height_path, 'r') as file:
            for _ in range(21):
                line = next(file).strip().replace('"', '').split(",")
    
                if line[0] == "XY calibration":
                    # Convert from um to mm
                    self.pixel_size = float(line[1]) * Sample.UM2MM
                if line[0] == "Horizontal":
                    self.horizontal = int(line[1])
                if line[0] == "Vertical":
                    self.vertical = int(line[1])
                if line[0] == "Unit":
                    self.unit = line[1]
        
        # Read height profile of sample from CSV
        height = np.genfromtxt(height_path, skip_header=23, delimiter=",", dtype=str)
        height = np.char.strip(height, '"')
        height[height == ''] = np.nan
        self.height = height.astype(float)

        # Read profilometer image from PNG file
        image_path = data_path / time / "profilometer" / f"{sample_id}.png"
        self.image = cv.imread(image_path, cv.IMREAD_COLOR)

In [3]:
def test_image_height_equality(path, ident):
    try:
        after = Sample(path, "after", ident)
    except FileNotFoundError:
        return None
        
    before = Sample(path, "before", ident)
    
    if not (before.image.shape[:2] == before.height.shape[:2]) and (after.image.shape[:2] == after.height.shape[:2]):
        return (ident, before.image.shape[:2], before.height.shape[:2], after.image.shape[:2], after.height.shape[:2])

In [4]:
# def test_image_height_equality(path, time, ident):
#     try:
#         after = Sample(path, time, ident)
#     except FileNotFoundError:
#         return None
        
#     if not np.all(np.argwhere(np.isnan(sample.height)) == np.argwhere(sample.image[:, :, 0] == 0)):
#         return ident

In [5]:
def test_sample_shape_equality(path, ident):
    try:
        after = Sample(path, "after", ident)
    except FileNotFoundError:
        return None
        
    before = Sample(path, "before", ident)
    
    if not (before.image.shape[:2] == before.height.shape[:2] == after.image.shape[:2] == after.height.shape[:2]):
        return (ident, before.image.shape[:2], after.image.shape[:2], before.height.shape[:2], after.height.shape[:2])

In [6]:
def find_unmatched_files(path, ident):
    try:
        before = Sample(path, "before", ident)
        after = Sample(path, "after", ident)
    except FileNotFoundError:
        return ident

In [7]:
path = Path("/data/dust/user/dammal/beegfs.migration/repos/ai2/data/processed")
time = "before"
sample_ids = [i.replace(".png", "") for i in os.listdir(path / time / "profilometer")]
ncpu = os.cpu_count()
circle_df = pd.read_csv(path / time / "sample-masking/circle-list/circle-list.csv", index_col = 0)

# Which samples have before image but no after image and vice versa

In [7]:
before_sample_ids = [i.replace(".png", "") for i in os.listdir(path / "before" / "profilometer")]
after_sample_ids = [i.replace(".png", "") for i in os.listdir(path / "after" / "profilometer")]

unmatched_before = Parallel(n_jobs=ncpu)(delayed(find_unmatched_files)(path, ident) for ident in before_sample_ids)
unmatched_after = Parallel(n_jobs=ncpu)(delayed(find_unmatched_files)(path, ident) for ident in after_sample_ids)

unmatched_before = [x for x in unmatched_before if x is not None]
unmatched_after = [x for x in unmatched_after if x is not None]

unmatched_before.sort()
unmatched_after.sort()

unmatched_before, unmatched_after

(['2-3-11-1',
  '2-3-11-2',
  '2-3-11-3',
  '2-3-11-4',
  '4-3-10-1',
  '4-3-10-2',
  '4-3-10-3',
  '4-3-10-4',
  '4-3-11-1',
  '4-3-11-2',
  '4-3-11-3',
  '4-3-11-4',
  '4-3-12-1',
  '4-3-12-2',
  '4-3-12-3',
  '4-3-12-4',
  '4-3-7-1',
  '4-3-7-2',
  '4-3-7-3',
  '4-3-7-4',
  '4-3-8-1',
  '4-3-8-2',
  '4-3-8-3',
  '4-3-8-4',
  '4-3-9-1',
  '4-3-9-2',
  '4-3-9-3',
  '4-3-9-4'],
 ['3-1-10-1',
  '3-1-10-2',
  '3-1-10-3',
  '3-1-10-4',
  '3-1-11-1',
  '3-1-11-2',
  '3-1-11-3',
  '3-1-11-4',
  '3-1-12-1',
  '3-1-12-2',
  '3-1-12-3',
  '3-1-12-4',
  '3-1-7-1',
  '3-1-7-2',
  '3-1-7-3',
  '3-1-7-4',
  '3-1-8-1',
  '3-1-8-2',
  '3-1-8-3',
  '3-1-8-4',
  '3-1-9-1',
  '3-1-9-2',
  '3-1-9-3',
  '3-1-9-4',
  '3-2-1-1',
  '3-2-1-2',
  '3-2-1-3',
  '3-2-1-4',
  '3-2-10-1',
  '3-2-10-2',
  '3-2-10-3',
  '3-2-10-4',
  '3-2-11-1',
  '3-2-11-2',
  '3-2-11-3',
  '3-2-11-4',
  '3-2-12-1',
  '3-2-12-2',
  '3-2-12-3',
  '3-2-12-4',
  '3-2-2-1',
  '3-2-2-2',
  '3-2-2-3',
  '3-2-2-4',
  '3-2-3-1',
  '3-2-3-2

# Which are samples where the size of the image does not agree to the height map

In [8]:
before_sample_ids = [i.replace(".png", "") for i in os.listdir(path / "before" / "profilometer")]

image_height_inequality = Parallel(n_jobs=ncpu)(delayed(test_image_height_equality)(path, ident) for ident in before_sample_ids)

image_height_inequality = [x for x in image_height_inequality if x is not None]
image_height_inequality.sort()
image_height_inequality

[('4-1-6-4', (1666, 1892), (1667, 1893), (1667, 1893), (1667, 1893)),
 ('4-3-1-1', (1666, 1892), (1667, 1893), (1667, 1893), (1667, 1893)),
 ('4-3-1-2', (1666, 1892), (1667, 1893), (1667, 1893), (1667, 1893)),
 ('4-3-1-3', (1666, 1892), (1667, 1894), (1667, 1893), (1667, 1893)),
 ('4-3-1-4', (1666, 1892), (1667, 1893), (1667, 1893), (1667, 1893))]

For some samples the height map shape does not agree with the image shape. That could be a potential problem since the height map might be shiftes against the image if centereing takes place

# Which samples which size of the image and heights are not equal for before and after images and height maps

In [9]:
before_sample_ids = [i.replace(".png", "") for i in os.listdir(path / "before" / "profilometer")]

sample_shape_inequality = Parallel(n_jobs=ncpu)(delayed(test_sample_shape_equality)(path, ident) for ident in before_sample_ids)

sample_shape_inequality = [x for x in sample_shape_inequality if x is not None]
sample_shape_inequality.sort()

sample_shape_inequality

[('1-1-11-1', (1668, 1601), (1667, 1601), (1668, 1601), (1667, 1601)),
 ('1-1-11-2', (1668, 1601), (1667, 1601), (1668, 1601), (1667, 1601)),
 ('1-1-11-4', (1668, 1601), (1667, 1601), (1668, 1601), (1667, 1601)),
 ('1-1-12-2', (1667, 1601), (1667, 1602), (1667, 1601), (1667, 1602)),
 ('1-1-2-1', (1668, 1601), (1667, 1601), (1668, 1601), (1667, 1601)),
 ('1-1-2-2', (1668, 1601), (1667, 1601), (1668, 1601), (1667, 1601)),
 ('1-1-2-3', (1668, 1601), (1667, 1601), (1668, 1601), (1667, 1601)),
 ('1-1-2-4', (1668, 1601), (1667, 1601), (1668, 1601), (1667, 1601)),
 ('1-1-5-1', (1666, 1601), (1667, 1601), (1666, 1601), (1667, 1601)),
 ('1-1-5-2', (1667, 1601), (1668, 1601), (1667, 1601), (1668, 1601)),
 ('1-1-5-3', (1667, 1601), (1668, 1601), (1667, 1601), (1668, 1601)),
 ('1-1-5-4', (1666, 1601), (1667, 1601), (1666, 1601), (1667, 1601)),
 ('1-1-9-1', (1667, 1601), (1668, 1601), (1667, 1601), (1668, 1601)),
 ('1-1-9-2', (1667, 1601), (1668, 1601), (1667, 1601), (1668, 1601)),
 ('1-1-9-3', (16